# BỐI CẢNH PHÂN TÍCH - GIAI ĐOẠN ỔN ĐỊNH 


## I. TỪ BÀI TOÁN QUẢN TRỊ ĐẾN CÂU HỎI PHÂN TÍCH 

Sau những ngày đầu chật vật với câu hỏi “có bán được hàng không”, Danny nhận ra một sự thật: **một đơn hàng chỉ thực sự có giá trị khi đến được tay khách hàng**. Bởi lẽ, dù bếp có làm ra những chiếc pizza ngon nhất, nếu khâu giao hàng chậm trễ, thái độ tài xế kém, hay đơn hàng bị hủy giữa chừng, thì tất cả công sức trước đó đều đổ sông đổ bể. Đây chính là thời điểm Danny phải **chuyển từ tư duy “bán được” sang tư duy “phục vụ tốt”**.

Nỗi trăn trở lớn nhất lúc này không còn là số lượng, mà là **chất lượng dịch vụ giao hàng**. Khách hàng phải chờ bao lâu kể từ lúc nhấn nút đặt đến khi cầm hộp pizza trên tay? Runner có đến lấy hàng đúng giờ không? Liệu có tài xế nào thường xuyên hủy đơn, làm xấu mặt thương hiệu? Và xa hơn, khi mùa cao điểm đến, Danny có cần tuyển thêm người hay không – hay đội ngũ hiện tại đã đủ sức gồng gánh?

Trả lời những câu hỏi ấy không chỉ giúp Danny **giữ chân khách hàng** trong giai đoạn nhạy cảm này, mà còn xây dựng nền tảng cho **văn hóa vận hành dựa trên dữ liệu**. Bởi vậy, toàn bộ phân tích của File B sẽ xoay quanh ba trụ cột: **con người (Runner), quy trình (Operations), và trải nghiệm (Customer)**.


| Câu hỏi quản trị | Các câu hỏi phân tích |
|------------------|-----------------------|
| Tuyển dụng tài xế có theo kịp tăng trưởng? | B.1 – Số runner đăng ký theo tuần |
| Bếp có làm chậm tiến độ giao hàng không? | B.2 – Thời gian trung bình đến lấy hàng |
| Số lượng pizza ảnh hưởng thế nào đến thời gian chuẩn bị? | B.3 – Ảnh hưởng của số pizza đến thời gian chuẩn bị |
| Chi phí giao hàng cho mỗi khách là bao nhiêu? | B.4 – Quãng đường giao trung bình mỗi khách |
| Mức độ ổn định của dịch vụ giao hàng ra sao? | B.5 – Chênh lệch thời gian giao nhanh nhất – chậm nhất |
| Runner nào đang hoạt động hiệu quả nhất? | B.6 – Tốc độ di chuyển trung bình từng runner |
| Runner nào cần được đào tạo lại? | B.7 – Tỉ lệ giao thành công của mỗi runner |



## II. PHÂN TÍCH VÀ ĐỀ XUẤT HÀNH ĐỘNG 

In [0]:
%sql
USE Pizza_Runner

### NHÓM 1: TUYỂN DỤNG VÀ NĂNG LỰC ĐỘI NGŨ GIAO HÀNG 

#### B.1. How many runners signed up for each 1 week period? 

(Số runner đăng ký theo tuần)


Trong SQL Server, với cấu hình máy chủ đặt tại Việt Nam (phiên bản mới, giao diện xanh), hệ thống mặc định xem **Chủ nhật** là ngày bắt đầu tuần – có thể kiểm tra qua biến `@@DATEFIRST` trả về giá trị 7. Ngược lại, Databricks SQL lại áp dụng quy ước quốc tế phổ biến hơn: tuần luôn bắt đầu từ **Thứ Hai**. Sự khác biệt này không chỉ ảnh hưởng đến thứ tự các ngày trong tuần, mà còn kéo theo một hệ quả quan trọng: **tuần đầu tiên của năm thường không trọn vẹn 7 ngày**. Chẳng hạn, nếu ngày 01/01/2021 rơi vào Thứ Sáu, thì theo cách tính của SQL Server, tuần 1 chỉ kéo dài đúng 2 ngày (Thứ Sáu và Thứ Bảy) trước khi bước sang tuần mới vào Chủ nhật. Tương tự, trên Databricks, tuần đầu tiên cũng có thể ngắn hơn nếu ngày đầu năm không phải Thứ Hai. Điều này dễ gây sai lệch khi so sánh số liệu theo tuần giữa các năm hoặc khi tổng hợp dữ liệu từ nhiều nguồn.

Chính vì thế, để đảm bảo tính nhất quán và phù hợp với thực tế vận hành của Pizza Runner, tôi không sử dụng cách tính tuần mặc định của cả hai hệ thống. Thay vào đó, tôi chọn **ngày 01/01/2021** làm mốc cố định. Từ mốc này, mỗi tuần được định nghĩa là một khối trọn vẹn **7 ngày liên tiếp**, bất kể thứ trong tuần. Như vậy, tuần 1 sẽ luôn bắt đầu từ 01/01 và kết thúc vào 07/01, tuần 2 từ 08/01 đến 14/01, và tiếp tục đều đặn. Cách làm này loại bỏ hoàn toàn sự phụ thuộc vào cấu hình máy chủ hay múi giờ, đồng thời tránh được tình trạng tuần đầu tiên bị “cụt” ngày – một ưu điểm then chốt khi cần phân tích xu hướng đều đặn theo chu kỳ 7 ngày.

In [0]:
%sql
-- Trong phân tích dữ liệu trường hợp này, không tính tuần theo cách tính hệ thống 
-- mà sẽ lấy cụm 7 ngày là 1 tuần
-- tức là tính từ ngày đầu tiên của tháng được tính là ngày thứ 1 của tuần thứ 1, thứ của ngày này sẽ được làm mốc để tính ngày bắt đầu cho các tuần sau của tháng 
-- cứ hết 7 ngày sẽ bắt đầu tuần mới 
SELECT 
    DATEADD(DAY, (DATEDIFF(DAY, '2021-01-01', registration_date) DIV 7) * 7, '2021-01-01') AS week_start_date,
    (DATEDIFF(DAY, '2021-01-01', r.registration_date) DIV 7) + 1 AS registration_week,
    COUNT(r.runner_id) AS total_runners_signed_up
FROM destination.runners r
GROUP BY 
    DATEADD(DAY, (DATEDIFF(DAY, '2021-01-01', registration_date) DIV 7) * 7, '2021-01-01'),
    (DATEDIFF(DAY, '2021-01-01', r.registration_date) DIV 7) + 1
ORDER BY registration_week;

week_start_date,registration_week,total_runners_signed_up
2021-01-01T00:00:00.000Z,1,2
2021-01-08T00:00:00.000Z,2,1
2021-01-15T00:00:00.000Z,3,1


### NHÓM 2: HIỆU SUẤT KHÂU BẾP VÀ ẢNH HƯỞNG ĐẾN GIAO HÀNG

#### B.2. What was the average time in minutes it took for each runner to arrive at the Pizza Runner HQ to pickup the order? 

(Thời gian trung bình runner đến lấy hàng)

In [0]:
%sql
-- CÁCH 1:
SELECT
		r.runner_id,
		AVG(DATEDIFF(MINUTE, o.order_time, r.pickup_time))	AS avg_pickup_minutes
FROM destination.orders o
INNER JOIN destination.runner_orders r ON o.order_id = r.order_id 
WHERE r.pickup_time IS NOT NULL 
GROUP BY r.runner_id 

runner_id,avg_pickup_minutes
1,14.0
3,10.0
2,19.666666666666668


In [0]:
%sql
-- CÁCH 2:

-- VIẾT NÂNG CAO HƠN 
-- nên thêm một dấu chấm phẩy ngay sát trước chữ WITH để chặn đứng mọi lỗi tiềm ẩn từ các câu lệnh bên trên
;WITH RECURSIVE PickupTime_CTE AS (
		SELECT
				r.runner_id,
				o.order_id,
				o.order_time ,
				r.pickup_time,
				-- Tính số phút chênh lệch cho từng đơn
				DATEDIFF(MINUTE, o.order_time, r.pickup_time) AS pickup_minutes
		FROM destination.orders o
		JOIN destination.runner_orders r ON o.order_id = r.order_id
		WHERE r.pickup_time IS NOT NULL 
			  AND r.cancellation IS NULL 
)
SELECT 
		p.runner_id ,
		ROUND(AVG(CAST(p.pickup_minutes AS FLOAT)),2) AS avg_pickup_minutes
FROM	PickupTime_CTE p
-- Có thể thêm điều kiện lọc nhiễu: WHERE pickup_minutes >= 0 (đề phòng lỗi log time ngược)
WHERE p.pickup_minutes >= 0
GROUP BY p.runner_id 
ORDER BY p.runner_id 

runner_id,avg_pickup_minutes
1,14.0
2,19.67
3,10.0


#### B.3. Is there any relationship between the number of pizzas and how long the order takes to prepare? 

(Ảnh hưởng của số pizza đến thời gian chuẩn bị)

In [0]:
%sql
-- Để tìm mối liên hệ giữa X (Số lượng Pizza) và Y (Thời gian chuẩn bị), ta bóc tách bài toán làm 3 chặng :
;WITH RECURSIVE Chang01_TinhThoiGian AS (
	-- Thời gian chuẩn bị tính từ lúc nhận đơn đến lúc runner_id nhận bánh đi giao 
	SELECT
			r.runner_id,
			o.customer_id,
			o.order_id,
			o.order_time,
			r.pickup_time,
			DATEDIFF(MINUTE, o.order_time, r.pickup_time ) AS Time_for_Prepare 
	FROM destination.orders o
	INNER JOIN destination.runner_orders r ON o.order_id = r.order_id

	-- đây là dự án demo nhỏ, nên nếu cancellation thì pickup_time NULL nên không cần tính 
	-- nhưng nếu dữ liệu lớn hơn, thời gian chuẩn bị cho đơn hảng bị hủy phải xem xét là hủy lúc làm xong hay chưa, nhiều khi xong rồi mà do lỗi trong quy trình nên không giao 
	-- hay những vấn đề khác 
	WHERE r.cancellation IS NULL

)
, Chang02_DemSoLuongBanh AS (
    SELECT
			order_id,
			COUNT(pizza_id) AS count_pizzas 
	FROM destination.customer_orders
	GROUP BY order_id 
)
-- CHẶNG 3: Kết hợp 2 bảng nháp lại để tìm Insight
SELECT
		c2.count_pizzas AS `Quy mô đơn hàng (Số bánh)`,
		COUNT(c1.order_id) AS `Tổng số đơn hàng`, -- Đếm số lượng đơn để xem mẫu đủ lớn không
		ROUND(AVG(CAST(c1.Time_for_Prepare AS FLOAT)), 2) AS `Thời gian chuẩn bị trung bình (Phút)`
FROM	Chang01_TinhThoiGian c1
INNER JOIN Chang02_DemSoLuongBanh c2 ON c1.order_id = c2.order_id
-- Bước gom nhóm quyết định: Gom theo quy mô đơn hàng (1 bánh, 2 bánh, 3 bánh...)
GROUP BY c2.count_pizzas
ORDER BY c2.count_pizzas;



Quy mô đơn hàng (Số bánh),Tổng số đơn hàng,Thời gian chuẩn bị trung bình (Phút)
1,5,12.0
2,2,18.0
3,1,29.0


### NHÓM 3: CHI PHÍ VÀ KHOẢNG CÁCH GIAO HÀNG 

#### B.4. What was the average distance travelled for each customer? 

(Quãng đường giao trung bình mỗi khách)

In [0]:
%sql
SELECT
		o.customer_id,
		ROUND(AVG(CAST(r.distance AS FLOAT)),2) AS avg_distance_by_cust
FROM destination.orders o
INNER JOIN destination.runner_orders r ON o.order_id = r.order_id
WHERE r.cancellation IS NULL 
GROUP BY o.customer_id 
ORDER BY avg_distance_by_cust ASC

customer_id,avg_distance_by_cust
104,10.0
102,18.4
101,20.0
103,23.4
105,25.0


### NHÓM 4: ĐỘ ỔN ĐINH VÀ CHẤT LƯỢNG GIAO HÀNG 

#### B.5. What was the difference between the longest and shortest delivery times for all orders? 

(Chênh lệch thời gian giao nhanh nhất – chậm nhất)

In [0]:
%sql
SELECT
		MAX(r.duration) AS the_longest_delivery_times,
		MIN(r.duration) AS the_shortest_delivery_times,
		MAX(r.duration) - MIN(r.duration) AS difference_time
FROM destination.runner_orders r
WHERE r.cancellation IS NULL 

the_longest_delivery_times,the_shortest_delivery_times,difference_time
40,10,30


### NHÓM 5: HIỆU SUẤT CÁ NHÂN TỪNG RUNNER

#### B.6. What was the average speed for each runner for each delivery and do you notice any trend for these values? 

(Tốc độ di chuyển trung bình từng runner)

In [0]:
%sql
SELECT
		r.runner_id,
		r.order_id,
		CAST(ROUND(r.distance, 2) AS DECIMAL(10,2)) AS distance_km,
		r.duration AS duration_mins , 
		-- s = v * t => v = s/t = km/(míns : 60) = km/ h --> v = s * 60 / mins (km/h)
		CAST(ROUND((r.distance * 60.0  / r.duration ),2) AS DECIMAL(10,2)) AS avg_speed_for_each_runner_for_each_delivery 
FROM destination.runner_orders r
WHERE r.cancellation IS NULL 
ORDER BY r.runner_id ASC,
		 r.order_id ASC 

runner_id,order_id,distance_km,duration_mins,avg_speed_for_each_runner_for_each_delivery
1,1,20.00,32,37.50
1,2,20.00,27,44.44
1,3,13.40,20,40.20
1,10,10.00,10,60.00
2,4,23.40,40,35.10
2,7,25.00,25,60.00
2,8,23.40,15,93.60
3,5,10.00,15,40.00


#### B.7. What is the successful delivery percentage for each runner? 

(Tỉ lệ giao thành công của mỗi runner)

In [0]:
%sql
;WITH RECURSIVE Runner_Stats AS (
    SELECT 
			r.runner_id,
			SUM(CASE WHEN r.cancellation IS NULL THEN 1 ELSE 0 END ) AS successful_delivery,
			SUM(CASE WHEN r.cancellation IS NOT NULL THEN 1 ELSE 0 END ) AS failed_delivery
	FROM destination.runner_orders r 
	GROUP BY r.runner_id 
)
SELECT 
    runner_id,
    successful_delivery,
    failed_delivery,
    -- Gọi tên 2 cột trên ra để ráp công thức tính % ở đây
	(successful_delivery * 100.0) / (successful_delivery + failed_delivery)  AS  successful_delivery_rate 
FROM Runner_Stats;


runner_id,successful_delivery,failed_delivery,successful_delivery_rate
1,4,0,100.00000000000000
2,3,1,75.00000000000000
3,1,1,50.00000000000000


## III. TỔNG KẾT 

## PHỤ LỤC: KHÁC BIỆT CÚ PHÁP GIỮA T‑SQL VÀ DATABRICKS SQL

Trong quá trình phân tích, một số câu lệnh yêu cầu điều chỉnh cú pháp khi chuyển đổi giữa hai môi trường. Bảng dưới đây tóm tắt các điểm khác biệt đã gặp trong file này. 

| Vấn đề | T‑SQL (SQL Server) | Databricks SQL (Spark SQL) | Lý do |
|--------|-------------------|----------------------------|-------|
| **Phép chia số nguyên** | `DATEDIFF(DAY, …) / 7` cho kết quả số nguyên (integer division) | Phải dùng `DATEDIFF(DAY, …) DIV 7` hoặc `FLOOR(... / 7)` | Spark trả về `DOUBLE` khi chia hai số nguyên; cần ép kiểu về số nguyên. |
| **CTE đệ quy** | `WITH cte AS (...)` không cần từ khóa `RECURSIVE` | `WITH RECURSIVE cte AS (...)` bắt buộc phải có `RECURSIVE` | Spark tuân thủ chuẩn ANSI SQL, yêu cầu khai báo rõ ràng CTE đệ quy. |
| **Alias chứa ký tự đặc biệt** | Cho phép dùng dấu ngoặc vuông: `[Quy mô đơn hàng (Số bánh)]` | Phải dùng dấu backtick: `` `Quy mô đơn hàng (Số bánh)` `` | Databricks không hỗ trợ bracket notation; backtick là quy chuẩn thay thế. |
| **Hàm `ROUND()` và kiểu trả về** | `ROUND(..., 2)` có thể trả về `DECIMAL` tuỳ biểu thức đầu vào | `ROUND(..., 2)` luôn trả về `DOUBLE`; muốn `DECIMAL(n,2)` cần `CAST` | `ROUND` trong Spark SQL chỉ làm tròn giá trị, không thay đổi kiểu dữ liệu. |